# Разные методы генерации из диффузионных моделей

Первый этап: воспроизводимый baseline с готовым DDIM и замороженной U-Net
`google/ddpm-cifar10-32`.

**Problem statement.** Как получить лучшее качество при одинаковом числе вызовов
нейросети, меняя сэмплер и расписание уровней шума? Позже реализуем DDIM,
DPM-Solver++ второго порядка и Heun и подберём расписание по MSE.

Здесь сначала проверим GPU, запустим маленький pilot, затем получим baseline:
5000 изображений, 100 NFE, время и FID относительно 50000 CIFAR-10 train.
Код готов к запуску; этот ноутбук ещё не содержит измеренных результатов T4.

В меню Colab выберите **Среда выполнения → Сменить среду выполнения → T4 GPU**.
Для ячеек загрузки модели, датасета и весов Inception нужен интернет.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Выберите GPU-среду перед запуском baseline'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM, GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)

## Загрузить исходники и подключить постоянное хранилище

Загрузите предоставленный архив **diffusion-sampling-colab.zip**, не всю папку
с датасетом или виртуальным окружением. На компьютере его можно пересобрать:
`python scripts/build_colab_bundle.py`.

Исходники распакуем в /content. Шумы, результаты и кэш сохраним в Google Drive,
чтобы продолжить запуск после отключения. Монтирование выполняется через ваш аккаунт.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile

drive.mount('/content/drive')
uploaded = files.upload()
archive = 'diffusion-sampling-colab.zip'
assert archive in uploaded, f'Выберите {archive}'
destination = Path('/content')
with ZipFile(archive) as bundle:
    for member in bundle.infolist():
        assert (destination / member.filename).resolve().is_relative_to(destination), 'Некорректный путь в архиве'
    bundle.extractall(destination)
PROJECT = destination / 'diffusion-sampling'
assert (PROJECT / 'pyproject.toml').is_file()
PERSISTENT = Path('/content/drive/MyDrive/llm-in-cv/diffusion-sampling')
PERSISTENT.mkdir(parents=True, exist_ok=True)
for name in ['data', 'results', '.cache']:
    target = PERSISTENT / name
    target.mkdir(exist_ok=True)
    link = PROJECT / name
    if not link.exists():
        link.symlink_to(target, target_is_directory=True)
print('Исходники:', PROJECT)
print('Постоянные данные:', PERSISTENT)

In [ ]:
%pip install -q -e /content/diffusion-sampling
%cd /content/diffusion-sampling

## Проверить окружение и код

doctor выводит версии и выбранное устройство. Тесты не скачивают модель и не
используют результаты синтетических сетей как baseline.
Все CLI-команды запускаются отдельным процессом; при ошибке ячейка остановится.

In [ ]:
import subprocess
import sys

def cli(*args):
    subprocess.run([sys.executable, '-m', 'diffusion_sampling', *args], check=True)

cli('doctor')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)

## Подготовить начальные шумы

Сохраняем сами тензоры, а не только seed. calibration, validation и итоговая
оценка разделены. Шум конкретного изображения не зависит от batch size.
При повторном выполнении ячейки готовые банки проверяются и используются снова.

In [ ]:
cli('prepare-noise', '--fid-count', '5000')

## Pilot на 16 изображениях

Используем DDIM, eta=0, 20 NFE. Цель — проверить, что генерация работает,
посмотреть результат и убедиться, что NFE посчитан правильно. Короткий pilot
не заменяет baseline, и его качество не характеризует длинную генерацию.

In [ ]:
cli('run', '--config', 'configs/pilot.json')
from IPython.display import Image, display
display(Image(filename='results/pilot_ddim_20/preview.png'))

## Полный батч baseline и оценка времени

Теперь DDIM 100 NFE, batch size 64. Один прогревочный батч не входит во время.
После одного измеренного батча команда остановится, сохранив результаты.
Если памяти не хватает, измените конфигурацию до нового запуска и задайте
новый output_dir. Не меняйте batch size внутри уже начатого эксперимента.

In [ ]:
cli('run', '--config', 'configs/baseline_ddim_100.json', '--max-batches', '1')
import json
summary_path = Path('results/baseline_ddim_100/summary.json')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
remaining = summary['requested_images'] - summary['generated_images']
print('Приблизительно осталось часов денойзинга:', remaining * summary['seconds_per_image'] / 3600)
display(Image(filename='results/baseline_ddim_100/preview.png'))

## Продолжить до 5000 изображений

Команда продолжает тот же эксперимент. Время учитывает только денойзинг,
а не запись в Drive и простои Colab. После прерывания подключите тот же Drive,
установите исходники и повторите команду с теми же настройками.
Если Colab изменил ключевые версии или устройство, используйте новый output_dir:
код не объединяет несопоставимые измерения.

In [ ]:
cli('run', '--config', 'configs/baseline_ddim_100.json')

## Посчитать FID отдельно

Скачиваются CIFAR-10 train и веса Inception, если их ещё нет. Метрика:
torch-fidelity 0.3.0, inception-v3-compat, 2048 признаков.
Сравниваем **5000 generated против 50000 real**. Не сравнивайте это число
напрямую с FID-50k из статьи. В папке images находятся только исходные 32×32 PNG;
увеличенный preview в метрику не попадает.

In [ ]:
cli('evaluate', '--run', 'results/baseline_ddim_100', '--device', 'cuda', '--download')

In [ ]:
summary = json.loads(summary_path.read_text())
fid_report = json.loads(Path('results/baseline_ddim_100/fid.json').read_text())
print('FID:', summary['fid'])
print('NFE на изображение:', summary['nfe_per_image'])
print('Суммарное время денойзинга, с:', summary['generation_seconds'])
print('Секунд на изображение:', summary['seconds_per_image'])
print('Численные метрики baseline готовы:', summary['baseline_metrics_complete'])
display(Image(filename='results/baseline_ddim_100/preview.png'))
print('Результаты и сырые тензоры сохранены в:', PERSISTENT / 'results/baseline_ddim_100')

## Что записать в отчёт РК 1

Сохраните конфигурацию, manifest, FID, NFE, время, размер обеих выборок и сетку
изображений. Опишите визуальное качество и заметные артефакты. Численные метрики
не заменяют визуальную проверку. Прикладывайте результаты реального запуска,
а не значения из примеров или тестов.

Дальше: собственный DDIM и сравнение шагов с Diffusers, затем DPM-Solver++ 2M,
Heun с DDPM/EDM-обвязкой, сравнение расписаний при одинаковом NFE и их подбор по MSE.
Подробности — README.md и docs/research_plan.md в архиве проекта.